In [1]:
import pandas as pd

df = pd.read_csv("Polonia Bytom_Pogo  Grodzisk Mazowiecki_4068759.csv", low_memory=False)

shots = df[df['event_type_name'] == 'Shot'].drop_duplicates(subset=['id']).copy()
shots = shots.sort_values(by=['index']).reset_index(drop=True)
shots[['minute', 'second', 'team_name', 'player_name', 'statsbomb_xg', 'outcome_name']].head()

,minute,second,team_name,player_name,statsbomb_xg,outcome_name
0,4,13,Polonia Bytom,Benedik Mioč,0.058596,Off T
1,5,6,Polonia Bytom,Kamil Wojtyra,0.100382,Blocked
2,13,38,Polonia Bytom,Kamil Wojtyra,0.028285,Off T
3,17,48,Polonia Bytom,Mikolaj Labojko,0.042174,Saved
4,21,13,Pogoń Grodzisk Mazowiecki,Kacper Los,0.197502,Goal


In [2]:
score = {'Polonia Bytom': 0, 'Pogoń Grodzisk Mazowiecki': 0}
results = []

for idx, row in shots.iterrows():
    team = row['team_name']
    opp = 'Pogoń Grodzisk Mazowiecki' if team == 'Polonia Bytom' else 'Polonia Bytom'
    
    if score[team] == score[opp]:
        state = 'Remis'
    elif score[team] > score[opp]:
        state = 'Wygrana'
    else:
        state = 'Przegrana'
        
    results.append({'Zespół': team, 'Stan meczu': state, 'xG': row['statsbomb_xg']})
    
    if row['outcome_name'] == 'Goal':
        score[team] += 1

results_df = pd.DataFrame(results)
xg_summary = results_df.groupby(['Zespół', 'Stan meczu'])['xG'].sum().reset_index()

pivot_xg = xg_summary.pivot(index='Stan meczu', columns='Zespół', values='xG').fillna(0)
pivot_xg['Różnica'] = pivot_xg['Pogoń Grodzisk Mazowiecki'] - pivot_xg['Polonia Bytom']
pivot_xg.columns.name = None
pivot_xg.index.name = None
pivot_xg = pivot_xg.reindex(['Wygrana', 'Remis', 'Przegrana'])
pivot_xg.round(2)

,Pogoń Grodzisk Mazowiecki,Polonia Bytom,Różnica
Wygrana,0.12,0.07,0.05
Remis,0.61,0.92,-0.31
Przegrana,0.02,0.08,-0.06
